In [ ]:
%pip install numpy mne osl-dynamics tensorflow ipyevents ipywidgets pyvista pyvistaqt pandas seaborn nibabel nilearn scikit-image numba pyyaml scikit-learn

In [3]:
import sys
!conda run -n hmmnet pip install h5io

'conda' is not recognized as an internal or external command,
operable program or batch file.


In [1]:
import mne
import numpy as np
import scipy.io
import h5py

from osl_dynamics import analysis, inference, files
from osl_dynamics.meeg import preproc, rhino, source_recon, parcellation
from osl_dynamics.data import Data
from osl_dynamics.utils import plotting
from osl_dynamics.utils.filenames import OSLFilenames


In [2]:
# =============================================================================
# 1. LOAD DATA FROM MAT FILE
# =============================================================================

subject_folder = '//rdp.arc.ucl.ac.uk/ritd-ag-project-rd01pz-dbush99/Katja Button Press Data/s05/'
subject_filename = 'Subject5_spm_mne'

filepath = subject_folder + subject_filename + '.mat'

In [3]:
# Extract the 240 trials and retain all channels.

def _read_h5_node(node):
    """Decode HDF5 groups and references while retaining numeric arrays."""
    if isinstance(node, h5py.Group):
        return {key: _read_h5_node(node[key]) for key in node}

    value = node[()]
    ref_type = h5py.check_dtype(ref=node.dtype)
    if ref_type is not None:
        refs = np.asarray(value)
        decoded = [
            _read_h5_node(f[ref]) if ref else None
            for ref in refs.flat
        ]
        return np.asarray(decoded, dtype=object).reshape(refs.shape)

    return np.asarray(value)


def _decode_matlab_string(value):
    """Decode a MATLAB character array stored as integer character codes."""
    chars = np.asarray(value).ravel()
    return "".join(chr(int(c)) for c in chars if int(c) != 0).strip()


with h5py.File(filepath, "r") as f:
    raw_data = np.asarray(f["data"])
    mat_data = raw_data.transpose(tuple(reversed(range(raw_data.ndim))))

    names_ds = f["chan_names"]
    chan_names_all = []
    for ref in names_ds[:, 0]:
        chan_names_all.append(_decode_matlab_string(f[ref][()]))

    # Keep the complete sensor structure for reference; do not discard fields
    # that cannot be represented directly in MNE.
    sens_metadata = _read_h5_node(f["sens"]) if "sens" in f else None

n_trials = 240
n_channels = len(chan_names_all)

if mat_data.ndim != 3:
    raise ValueError(f"Expected 3-D epoched data, got shape {mat_data.shape}")

channel_axes = [i for i, size in enumerate(mat_data.shape) if size == n_channels]
trial_axes = [i for i, size in enumerate(mat_data.shape) if size == n_trials]

if len(channel_axes) != 1 or len(trial_axes) != 1 or channel_axes[0] == trial_axes[0]:
    raise ValueError(
        f"Cannot uniquely identify channel/trial axes in shape {mat_data.shape}; "
        f"{n_channels} channels and {n_trials} trials were expected."
    )

time_axis = next(i for i in range(3) if i not in (channel_axes[0], trial_axes[0]))
epochs_data = np.moveaxis(
    mat_data,
    (trial_axes[0], channel_axes[0], time_axis),
    (0, 1, 2),
)

# Infer common types from names; unknown channels remain present as misc.
def _infer_channel_type(name):
    upper = name.upper()
    if upper.startswith("EEG"):
        return "eeg"
    if upper.startswith("EMG"):
        return "emg"
    if upper.startswith("EOG"):
        return "eog"
    if upper.startswith("ECG"):
        return "ecg"
    if upper.startswith("STI") or upper.startswith("TRIG"):
        return "stim"
    if upper.startswith("MEG"):
        return "grad" if "GRAD" in upper else "mag"
    return "misc"

channel_types = [_infer_channel_type(name) for name in chan_names_all]

# Optional: fill this using verified sensor metadata.
# Values are (position_xyz, orientation_xyz), in metres and device coordinates.
meg_geometry = {}

info = mne.create_info(
    ch_names=chan_names_all,
    sfreq=1000.0,
    ch_types=channel_types,
)

# Fill channel locations and orientations for MEG channels if available.
for idx, name in enumerate(chan_names_all):
    if name in meg_geometry:
        position, orientation = meg_geometry[name]
        info["chs"][idx]["loc"][:6] = np.r_[position, orientation]

# Create an MNE EpochsArray object from the data and info.
epochs_all = mne.EpochsArray(epochs_data, info, verbose=False)

# Keep source channel metadata paired with the Epochs object.
channel_metadata = {
    "channel_names": chan_names_all,
    "channel_types": channel_types,
    "sens": sens_metadata,
}

print(f"Epochs shape (trials, channels, times): {epochs_data.shape}")
print(f"Channel types: {dict(zip(chan_names_all, channel_types))}")
print("Sensor metadata retained in `channel_metadata['sens']`.")

Epochs shape (trials, channels, times): (240, 341, 14801)
Channel types: {'UPPT001': 'misc', 'UPPT002': 'misc', 'SCLK01': 'misc', 'BG1': 'misc', 'BG2': 'misc', 'BG3': 'misc', 'BP1': 'misc', 'BP2': 'misc', 'BP3': 'misc', 'BR1': 'misc', 'BR2': 'misc', 'BR3': 'misc', 'G11': 'misc', 'G12': 'misc', 'G13': 'misc', 'G22': 'misc', 'G23': 'misc', 'P11': 'misc', 'P12': 'misc', 'P13': 'misc', 'P22': 'misc', 'P23': 'misc', 'Q11': 'misc', 'Q12': 'misc', 'Q13': 'misc', 'Q22': 'misc', 'Q23': 'misc', 'R11': 'misc', 'R12': 'misc', 'R13': 'misc', 'R22': 'misc', 'R23': 'misc', 'MLC11': 'misc', 'MLC12': 'misc', 'MLC13': 'misc', 'MLC14': 'misc', 'MLC15': 'misc', 'MLC16': 'misc', 'MLC17': 'misc', 'MLC21': 'misc', 'MLC22': 'misc', 'MLC23': 'misc', 'MLC24': 'misc', 'MLC25': 'misc', 'MLC31': 'misc', 'MLC32': 'misc', 'MLC41': 'misc', 'MLC42': 'misc', 'MLC51': 'misc', 'MLC52': 'misc', 'MLC53': 'misc', 'MLC54': 'misc', 'MLC55': 'misc', 'MLC61': 'misc', 'MLC62': 'misc', 'MLC63': 'misc', 'MLF11': 'misc', 'MLF12': '

In [ ]:
with h5py.File(filepath, 'r') as f:
    data       = np.array(f['data']).T        # (n_channels, n_times)
    sfreq      = float(np.array(f['fs']).flatten()[0])
    chan_names = []
    for ref in f['chan_names'][:, 0]:
        name = ''.join(chr(c) for c in f[ref][:].flatten())
        chan_names.append(name)
    print(f['sens'])



print(f"Data shape: {data.shape}")
print(f"N channels: {len(chan_names)}")
print(f"Sampling rate: {sfreq} Hz")

In [ ]:
# isolate EEG data from total channels
eeg_idx = [i for i, name in enumerate(chan_names) if name.startswith("EEG")]
eeg_names = [chan_names[i] for i in eeg_idx]
data_eeg = data[eeg_idx, :]

print(f"EEG data shape: {data_eeg.shape}")
print(f"EEG channel count: {len(eeg_names)}")
print(f"EEG data shape: {data_eeg.shape}")

# reorder data_eeg to have shape (n_epochs/trials, n_channels, n_times)#
data_eeg = data_eeg.transpose(1, 0, 2)
print("Reordered data_eeg shape:", data_eeg.shape)
print(f"EEG channel count: {len(eeg_names)}")
print(f"EEG data shape: {data_eeg.shape}")

# create MNE info object for EEG data
info = mne.create_info(
    ch_names=eeg_names,
    sfreq=sfreq,
    ch_types=["eeg"] * len(eeg_names),
)

# create MNE Epoch Array for EEG data (with shape (n_epochs, n_channels, n_times))
Epochs_data_array = mne.EpochsArray(data_eeg, info)
print(f"dev_head_t: {Epochs_data_array.info['dev_head_t']}")

In [4]:
# =============================================================================
# 3. SAVE
# =============================================================================
import os

os.chdir(subject_folder)        
outfile = subject_folder + subject_filename + '.fif'
epochs_all.save(outfile, overwrite=True)
print("Saved!")


C:\Users\marce\AppData\Local\Temp\ipykernel_39816\876772141.py:8: RuntimeWarning: This filename (//rdp.arc.ucl.ac.uk/ritd-ag-project-rd01pz-dbush99/Katja Button Press Data/s05/Subject5_spm_mne.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs_all.save(outfile, overwrite=True)


Overwriting existing file.
Splitting into 3 parts
Overwriting existing file.
Overwriting existing file.


OSError: [Errno 22] Invalid argument